# Lezione 06 — Stima e intervalli di confidenza

**Corso:** Statistica Applicata con Python · **Blocco 3:** Inferenza
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/06_estimacion_intervalos_confianza_IT.md`

## Cosa vedrai in questa lezione

- **Stimatore** e **stima**: la media di un campione come approssimazione della media della popolazione.
- La **distribuzione campionaria** e l'**errore standard** della media (σ/√n, e s/√n quando σ è sconosciuta).
- L'**intervallo di confidenza (IC)** per la media con la t di Student, con un calcolo a mano.
- L'**interpretazione corretta** dell'IC, verificata con un esperimento di **copertura**.
- Come cambia l'ampiezza dell'IC con il livello di confidenza e con la dimensione del campione.
- Il **bootstrap percentile** per la media e la mediana, e i suoi **limiti**.
- Un cenno all'IC di una **proporzione** (Wilson).

## Prerequisiti

- Lezione 1 (variabilità campionaria e distorsione) e lezione 5 (t di Student, teorema del limite centrale).
- Librerie: `numpy`, `pandas`, `matplotlib`, `scipy`, `statsmodels` (`pip install numpy pandas matplotlib scipy statsmodels`; sono già installate in Google Colab).

> **Come usare questo notebook:** esegui le celle in ordine. Prima di ogni esperimento scrivi cosa ti aspetti che accada. I tempi di consegna sono **sintetici** (*esempio illustrativo*): non sono dati reali di nessuna azienda.

---
## Cella 0 — Setup

Importiamo le librerie e fissiamo il **seme** del generatore casuale: con lo stesso seme, tutti otteniamo gli stessi numeri.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)          # seme fisso: tutti otteniamo gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerie caricate correttamente.")

---
## 1. Una popolazione nota per poter verificare tutto

Per *vedere* cosa fa uno stimatore dobbiamo conoscere la verità. Generiamo una **popolazione sintetica** di 100 000 ordini con il loro tempo di consegna in ore: la distribuzione è **asimmetrica** (molte consegne rapide e una coda di consegne lente), come spesso accade con i tempi di attesa. Nella vita reale non conosciamo mai la popolazione; qui sì, e per questo possiamo verificare se i nostri intervalli centrano il bersaglio.

In [ ]:
poblacion = rng.lognormal(mean=np.log(30), sigma=0.5, size=100_000)     # ore (esempio illustrativo)
MU = poblacion.mean()          # parametro: la media vera
SIGMA = poblacion.std()        # parametro: la deviazione standard vera
MEDIANA_POB = np.median(poblacion)

print(f"Media vera μ          = {MU:.2f} h")
print(f"Mediana vera          = {MEDIANA_POB:.2f} h")
print(f"Deviazione standard vera σ = {SIGMA:.2f} h")
print(f"Asimmetria (skew)           = {stats.skew(poblacion):.2f}")

plt.hist(poblacion, bins=80, range=(0, 150), color="steelblue", alpha=0.8)
plt.axvline(MU, color="crimson", linestyle="--", label="media")
plt.axvline(MEDIANA_POB, color="black", linestyle=":", label="mediana")
plt.title("Popolazione sintetica di tempi di consegna (esempio illustrativo)")
plt.xlabel("Ore"); plt.ylabel("Ordini"); plt.legend()
plt.show()

**Osserva:** la media (≈ 34 h) è a destra della mediana (≈ 30 h) a causa della coda lunga. Ricorda dalla lezione 1 che un campione è solo una parte della popolazione: la media di un campione sarà un numero simile a μ, ma non identico.

---
## 2. Stimatore, stima e distribuzione campionaria

Lo **stimatore** è la regola (qui, "calcola la media del campione"); la **stima** è il numero che risulta da *un* campione concreto. Estraiamo un campione di 50 ordini e poi ripetiamo il campionamento 2 000 volte per vedere come varia la stima: questa è la **distribuzione campionaria** della media (lezione 1, ora la quantifichiamo).

In [ ]:
N = 50
una_muestra = rng.choice(poblacion, size=N)
print(f"Stima con UN campione: media = {una_muestra.mean():.2f} h   (μ = {MU:.2f} h)")

medias = rng.choice(poblacion, size=(2_000, N)).mean(axis=1)       # 2 000 campioni de 50
print(f"Media delle 2 000 medie        : {medias.mean():.2f} h")
print(f"Deviazione standard delle 2 000 medie: {medias.std(ddof=1):.2f} h   ← errore standard empirico")
print(f"σ/√n = {SIGMA / np.sqrt(N):.2f} h                       ← errore standard teorico")

plt.hist(medias, bins=40, color="darkorange", alpha=0.85)
plt.axvline(MU, color="crimson", linestyle="--", label="μ")
plt.title("Distribuzione campionaria della media (n = 50, 2 000 campioni)")
plt.xlabel("Media del campione (h)"); plt.ylabel("Frequenza"); plt.legend()
plt.show()

**Cosa osservi?**

- La distribuzione delle medie è centrata su μ? Come si chiama questa proprietà (lezione 1)?
- La popolazione è molto asimmetrica; e la distribuzione delle medie? Collegalo al teorema del limite centrale (lezione 5).
- L'errore standard empirico e quello teorico σ/√n coincidono?

*(La tua risposta qui.)*

### L'errore standard e la dimensione del campione

Ripetiamo l'esperimento con n diversi. Nella pratica abbiamo **un solo** campione, quindi σ si stima con la deviazione standard campionaria s (con `ddof=1`) e l'errore standard stimato è s/√n.

In [ ]:
filas = []
for n in [10, 40, 160, 640]:
    muestras = rng.choice(poblacion, size=(2_000, n))
    filas.append({"n": n,
                  "errore standard empirico": muestras.mean(axis=1).std(ddof=1),
                  "teorico σ/√n": SIGMA / np.sqrt(n),
                  "stimato con 1 campione s/√n": muestras[0].std(ddof=1) / np.sqrt(n)})
display(pd.DataFrame(filas).set_index("n").round(2))

**Osserva:** ogni volta che moltiplichi n per 4, l'errore standard si dimezza (non si divide per 4). Per dimezzare l'errore bisogna quadruplicare i dati: i rendimenti sono decrescenti.

---
## 3. L'intervallo di confidenza per la media: un calcolo a mano

Un IC al 95 % per la media è **media ± t · s/√n**, dove t è il quantile 0,975 della t di Student con n − 1 gradi di libertà. Lo calcoliamo passo per passo con 8 tempi di consegna (*esempio illustrativo*), così puoi ripeterlo con carta e penna.

In [ ]:
pequena = np.array([24, 31, 28, 45, 27, 52, 30, 36])          # 8 tempi di consegna, in ore
n = len(pequena)
media = pequena.mean()
s = pequena.std(ddof=1)
ee = s / np.sqrt(n)
t_crit = stats.t.ppf(0.975, df=n - 1)
semiancho = t_crit * ee

print(f"n = {n}   media = {media:.3f}   s = {s:.3f}   s/√n = {ee:.3f}")
print(f"t(0,975; {n - 1} g.d.l.) = {t_crit:.3f}   semi-ampiezza = {semiancho:.3f}")
print(f"IC 95 % a mano        : ({media - semiancho:.2f}, {media + semiancho:.2f})")

ic_scipy = stats.t.interval(0.95, df=n - 1, loc=media, scale=ee)
print(f"IC 95 % con scipy     : ({ic_scipy[0]:.2f}, {ic_scipy[1]:.2f})")

z_crit = stats.norm.ppf(0.975)
print(f"IC con z (SBAGLIATO con n = 8): ({media - z_crit * ee:.2f}, {media + z_crit * ee:.2f})   ← troppo stretto")

**Osserva:** la formula a mano e `scipy` danno lo stesso risultato. Con soli 8 dati, t = 2,365 è nettamente maggiore dell'1,96 della normale: la t "paga" l'incertezza aggiunta dall'aver stimato σ con s. Usare z qui produrrebbe un intervallo troppo stretto.

---
## 4. Esperimento guidato: cosa significa "95 % di confidenza"?

Estraiamo **1 000 campioni** da 50 ordini, costruiamo **1 000 IC al 95 %** e contiamo quanti contengono la media vera μ. Poiché qui conosciamo μ, possiamo verificarlo. Prima di eseguire, scrivi quale proporzione ti aspetti.

In [ ]:
N_REP, N_MUESTRA, NIVEL = 1_000, 50, 0.95

muestras = rng.choice(poblacion, size=(N_REP, N_MUESTRA))
medias = muestras.mean(axis=1)
ees = muestras.std(axis=1, ddof=1) / np.sqrt(N_MUESTRA)
t_crit = stats.t.ppf(1 - (1 - NIVEL) / 2, df=N_MUESTRA - 1)
inferior, superior = medias - t_crit * ees, medias + t_crit * ees

contiene = (inferior <= MU) & (MU <= superior)
cobertura = contiene.mean()
ee_cob = np.sqrt(NIVEL * (1 - NIVEL) / N_REP)

print(f"Intervalli che contengono μ: {contiene.sum()} de {N_REP}")
print(f"Copertura osservata            : {cobertura:.3f}   (nominale {NIVEL}; errore standard della copertura ≈ {ee_cob:.3f})")

# i primi 50 intervalli, con quelli che mancano evidenziati
fig, ax = plt.subplots(figsize=(8, 6))
for i in range(50):
    color = "seagreen" if contiene[i] else "crimson"
    ax.hlines(i, inferior[i], superior[i], color=color, linewidth=2)
    ax.plot(medias[i], i, "o", color=color, markersize=3)
ax.axvline(MU, color="black", linestyle="--", label="μ (vera)")
ax.set_title("50 intervalli di confidenza al 95 %: in rosso, quelli che NON contengono μ")
ax.set_xlabel("Ore"); ax.set_ylabel("Campione numero"); ax.legend()
plt.show()

**Cosa osservi?**

- La copertura osservata è vicina a 0,95? Con dati asimmetrici e n = 50 di solito risulta un po' inferiore: la t presuppone una distribuzione della media quasi normale, e con questa coda lunga è solo approssimata.
- Nel grafico, quanti dei 50 intervalli falliscono? Si possono distinguere dagli altri *senza conoscere μ*?
- Per **un** intervallo concreto già calcolato, μ è dentro oppure no: in quel caso non esiste un "95 % di probabilità". A cosa si riferisce allora il 95 %?

*(La tua risposta qui.)*

---
## 5. Cosa rende un intervallo più ampio o più stretto

Due leve: il **livello di confidenza** (più confidenza, intervallo più ampio) e la **dimensione del campione** (più dati, più stretto). Lo vediamo con un campione fisso e poi facendo la media su molti.

In [ ]:
muestra = poblacion[:50]                                           # un campione fisso
ee_m = muestra.std(ddof=1) / np.sqrt(len(muestra))
filas = []
for nivel in [0.90, 0.95, 0.99]:
    lo, hi = stats.t.interval(nivel, df=len(muestra) - 1, loc=muestra.mean(), scale=ee_m)
    filas.append({"livello": nivel, "limite inferiore": lo, "limite superiore": hi, "ampiezza": hi - lo})
display(pd.DataFrame(filas).set_index("livello").round(2))

anchos = []
for n in [10, 20, 40, 80, 160, 320]:
    m = rng.choice(poblacion, size=(500, n))
    semi = stats.t.ppf(0.975, n - 1) * m.std(axis=1, ddof=1) / np.sqrt(n)
    anchos.append(2 * semi.mean())
plt.plot([10, 20, 40, 80, 160, 320], anchos, "o-", label="ampiezza media dell'IC 95 %")
plt.xscale("log"); plt.yscale("log")
plt.title("L'ampiezza dell'IC scende come 1/√n")
plt.xlabel("n"); plt.ylabel("Ampiezza (h)"); plt.legend()
plt.show()

**Osserva:** l'IC al 99 % è più ampio di quello al 95 %, che a sua volta è più ampio di quello al 90 %. Un intervallo più stretto **non è "migliore"**: a parità di dati, per stringerlo basta accettare meno confidenza. L'unico modo di stringerlo *senza* perdere confidenza è procurarsi più dati.

---
## 6. Bootstrap percentile

Per la media esiste una formula; per la **mediana** non c'è una formula semplice. Il *bootstrap* usa il campione stesso come sostituto della popolazione: **ricampiona con reinserimento** n dati del campione, calcola la statistica e ripete 10 000 volte. I percentili 2,5 e 97,5 di quei valori formano l'IC. Lo vettorizziamo: una matrice di indici di dimensione (B, n) evita i cicli.

In [ ]:
def bootstrap_dist(datos, estadistico, B, rng):
    """Distribuzione bootstrap: B ricampionamenti con reinserimento."""
    idx = rng.integers(0, len(datos), size=(B, len(datos)))
    return estadistico(datos[idx], axis=1)

def ic_percentil(dist, nivel=0.95):
    """IC percentile: quantili alfa/2 e 1 - alfa/2 della distribuzione bootstrap."""
    alfa = 1 - nivel
    return np.quantile(dist, [alfa / 2, 1 - alfa / 2])

muestra = rng.choice(poblacion, size=50)
B = 10_000
dist_media = bootstrap_dist(muestra, np.mean, B, rng)
dist_mediana = bootstrap_dist(muestra, np.median, B, rng)

ic_boot_media = ic_percentil(dist_media)
ic_boot_mediana = ic_percentil(dist_mediana)
ee_m = muestra.std(ddof=1) / np.sqrt(len(muestra))
ic_t = stats.t.interval(0.95, df=len(muestra) - 1, loc=muestra.mean(), scale=ee_m)
media_m, mediana_m = muestra.mean(), np.median(muestra)

print(f"Media del campione   = {media_m:.2f}   IC t = ({ic_t[0]:.2f}, {ic_t[1]:.2f})   IC bootstrap = ({ic_boot_media[0]:.2f}, {ic_boot_media[1]:.2f})")
print(f"Mediana del campione = {mediana_m:.2f}   IC bootstrap = ({ic_boot_mediana[0]:.2f}, {ic_boot_mediana[1]:.2f})")
print(f"Verità nella popolazione: μ = {MU:.2f}, mediana = {MEDIANA_POB:.2f}")

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
for ax, dist, ic, titulo, real in [(ejes[0], dist_media, ic_boot_media, "Bootstrap della media", MU),
                                   (ejes[1], dist_mediana, ic_boot_mediana, "Bootstrap della mediana", MEDIANA_POB)]:
    ax.hist(dist, bins=50, color="steelblue", alpha=0.85)
    ax.axvline(ic[0], color="black", linestyle="--"); ax.axvline(ic[1], color="black", linestyle="--")
    ax.axvline(real, color="crimson", label="valore vero")
    ax.set_title(titulo); ax.set_xlabel("Ore"); ax.legend()
plt.tight_layout(); plt.show()

**Cosa osservi?**

- L'IC bootstrap della media assomiglia a quello della t? Perché è ragionevole che si somiglino?
- La distribuzione bootstrap della mediana è più "frastagliata" o irregolare di quella della media? (Suggerimento: la mediana di n dati può assumere solo valori già presenti nel campione.)
- I due intervalli contengono il valore vero della popolazione in questo campione? Ricorda che questo, su un solo campione, è questione di fortuna al 95 %.

*(La tua risposta qui.)*

---
## 7. Bootstrap contro t: copertura e limiti

Verifichiamo per simulazione la copertura della **t** e del **bootstrap percentile** per la media, con la popolazione asimmetrica, in tre dimensioni di campione. A ogni ripetizione: un campione nuovo, un IC t e un IC bootstrap (1 000 ricampionamenti). Usiamo 500 ripetizioni, quindi l'errore standard di ogni copertura è circa 0,01.

In [ ]:
def cobertura_media(n, reps, B, rng):
    """Copertura dell'IC t e dell'IC bootstrap percentile per la media."""
    aciertos_t = aciertos_b = 0
    t_crit = stats.t.ppf(0.975, n - 1)
    for _ in range(reps):
        m = rng.choice(poblacion, size=n)
        semi = t_crit * m.std(ddof=1) / np.sqrt(n)
        aciertos_t += (m.mean() - semi <= MU <= m.mean() + semi)
        lo, hi = ic_percentil(bootstrap_dist(m, np.mean, B, rng))
        aciertos_b += (lo <= MU <= hi)
    return aciertos_t / reps, aciertos_b / reps

filas = []
for n in [10, 30, 100]:
    cob_t, cob_b = cobertura_media(n, 500, 1_000, rng)
    filas.append({"n": n, "copertura IC t": cob_t, "copertura IC bootstrap": cob_b})
display(pd.DataFrame(filas).set_index("n").round(3))

**Cosa osservi?**

- Con n = 10, nessuno dei due metodi raggiunge il 95 %. Quale resta più corto? Il bootstrap non "sistema" un campione molto piccolo: conosce solo i 10 dati che ha.
- Cosa succede al crescere di n?
- Conclusione: con dati asimmetrici e pochi dati, un IC "al 95 %" può coprire parecchio meno; bisogna dirlo e, se possibile, ampliare il campione.

*(La tua risposta qui.)*

### Il bootstrap non corregge un campione distorto

Simuliamo un registro che **conserva solo le consegne sotto le 40 ore** (per esempio perché le altre sono gestite in un altro sistema). È una distorsione di selezione (lezione 1): il bootstrap darà un intervallo molto "sicuro di sé" nel posto sbagliato.

In [ ]:
registro = poblacion[poblacion < 40]
muestra_sesgada = rng.choice(registro, size=200)
ic_sesgado = ic_percentil(bootstrap_dist(muestra_sesgada, np.mean, 10_000, rng))

print(f"Media del campione distorto: {muestra_sesgada.mean():.2f} h")
print(f"IC bootstrap 95 %              : ({ic_sesgado[0]:.2f}, {ic_sesgado[1]:.2f})")
print(f"Media vera della popolazione: {MU:.2f} h   →   dentro l'IC? {ic_sesgado[0] <= MU <= ic_sesgado[1]}")

**Osserva:** l'intervallo è stretto (200 dati) ma lontano da μ. Né la t né il bootstrap correggono una distorsione di selezione: un IC misura solo l'incertezza **campionaria**, non quella introdotta da una cattiva raccolta dei dati.

---
## 8. (Opzionale) IC per una proporzione: Wilson

Per una proporzione (per esempio la frazione di consegne che impiegano più di 48 h) l'intervallo più raccomandabile è quello di **Wilson**. Ci servirà nelle lezioni 7 e 13. `statsmodels` lo calcola con `proportion_confint`; lo confrontiamo con la formula normale semplice (Wald).

In [ ]:
from statsmodels.stats.proportion import proportion_confint

p_real = (poblacion > 48).mean()
muestra_p = rng.choice(poblacion, size=200)
x_tardias = int((muestra_p > 48).sum())
print(f"Proporzione vera di consegne > 48 h: {p_real:.3f}")
print(f"Campione: {x_tardias} su 200 → proporzione {x_tardias / 200:.3f}")

for metodo in ["wilson", "normal"]:
    lo, hi = proportion_confint(x_tardias, 200, alpha=0.05, method=metodo)
    print(f"  {metodo:7s} → ({lo:.3f}, {hi:.3f})")

print()
print("Caso estremo: 1 consegna in ritardo su 20")
p_hat = 1 / 20
lim_inf_wald = p_hat - 1.96 * np.sqrt(p_hat * (1 - p_hat) / 20)
print(f"  formula normale a mano → limite inferiore = {lim_inf_wald:.3f}   ← negativo!")
print(f"  Wilson → {tuple(round(v, 3) for v in proportion_confint(1, 20, alpha=0.05, method='wilson'))}")

**Osserva:** con una proporzione piccola e pochi dati, la formula normale semplice arriva a dare un limite inferiore negativo, impossibile per una proporzione. Wilson non ha questo problema e ha una copertura migliore.

---
## 9. Riepilogo della lezione

- Uno **stimatore** è una regola; una **stima**, il numero di un campione concreto. La sua variabilità si riassume nell'**errore standard**: σ/√n, stimato con s/√n.
- L'**IC per la media** con la t è media ± t · s/√n. Si usa t (non z) perché σ è stimata con s, soprattutto con n piccolo.
- Il **95 % descrive la procedura**: su campioni ripetuti, ≈ 95 % degli intervalli contiene μ. Non è la probabilità che μ stia in *questo* intervallo.
- Più confidenza → intervallo più ampio; più dati → più stretto (come 1/√n). Stretto non significa migliore.
- Il **bootstrap percentile** dà IC per statistiche senza formula semplice (come la mediana), ma non crea informazione: fallisce con campioni molto piccoli e non corregge le distorsioni.
- Per una **proporzione**, Wilson (`proportion_confint`) è preferibile alla formula normale semplice.
- Un IC **non è un test**: confrontare due gruppi guardando se gli intervalli si sovrappongono è una scorciatoia pericolosa (lezione 7).

**Prossimo passo:** il notebook di pratica di questa lezione e, poi, la lezione 7 (test d'ipotesi I).